# Reproducing the paper's results

This notebook runs every analysis script in the order of the paper and shows its output under each
heading. All values are computed from the saved model outputs in `results/` with the shared metrics in
`src/metrics.py`; no GPU, API key or restricted data is needed. Running it takes about three minutes.

Conventions: QWK on the 0-10 rubric total (full label list); MAE, ±1 accuracy and bias on the 0-5 final
grade; intervals and p-values from a paired question-cluster bootstrap (17 test questions, B = 10,000,
seed 42); Bonferroni alpha = 0.05/9 for the nine comparisons per metric.

In [1]:
import subprocess, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

def run(script):
    r = subprocess.run([sys.executable, str(ROOT / 'scripts' / script)], cwd=ROOT, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    assert r.returncode == 0, script

## Table 8. Zero-shot model selection (non-test data, N = 2,040)

Phase 1 replies re-read with one rule (decimal comma, rounding, clipping); see CHANGELOG.md. The `ties=even` rows are the reported values; `ties=up` is a sensitivity check.

In [2]:
run('table08_zeroshot_reread.py')

DeepSeek-7B-Chat     ties=even changed= 804 pm1=47.1 macro=49.4 QWK=0.435 r=0.550 MAE=1.428 bias=-0.963 P=49.2 R=72.2 incons=0.7 crit={'Clarity': 0.381, 'Terminology': 0.45, 'Coverage': 0.465, 'Accuracy': 0.243}
DeepSeek-7B-Chat     ties=up   changed= 808 pm1=47.2 macro=49.5 QWK=0.436 r=0.550 MAE=1.427 bias=-0.961 P=49.2 R=72.2 incons=0.7 crit={'Clarity': 0.381, 'Terminology': 0.45, 'Coverage': 0.465, 'Accuracy': 0.245}
Llama3-8B-Instruct   ties=even changed= 168 pm1=73.8 macro=71.9 QWK=0.762 r=0.764 MAE=0.853 bias=+0.131 P=82.9 R=27.0 incons=0.0 crit={'Clarity': 0.655, 'Terminology': 0.639, 'Coverage': 0.682, 'Accuracy': 0.719}
Llama3-8B-Instruct   ties=up   changed= 180 pm1=73.8 macro=71.8 QWK=0.761 r=0.763 MAE=0.854 bias=+0.135 P=82.9 R=27.0 incons=0.0 crit={'Clarity': 0.655, 'Terminology': 0.639, 'Coverage': 0.68, 'Accuracy': 0.719}
Gemma2-9B-IT         ties=even changed=   0 pm1=78.4 macro=76.5 QWK=0.819 r=0.820 MAE=0.726 bias=-0.039 P=71.7 R=84.9 incons=21.0 crit={'Clarity': 0.63

## Figures 5-6. Zero-shot grade distributions and per-criterion QWK

Writes the two figures to `figures/`.

In [3]:
run('figures/fig05_06_zeroshot.py')

non-test answers per model: {'DeepSeek-7B': 2040, 'LLaMA 3-8B': 2040, 'Gemma 2-9B': 2040}
DeepSeek-7B  per-criterion QWK: Clarity 0.381, Terminology 0.450, Coverage 0.465, Accuracy 0.243
LLaMA 3-8B   per-criterion QWK: Clarity 0.655, Terminology 0.639, Coverage 0.682, Accuracy 0.719
Gemma 2-9B   per-criterion QWK: Clarity 0.632, Terminology 0.586, Coverage 0.748, Accuracy 0.772
written: figures/fig_zeroshot_grade_distributions.pdf, figures/fig_zeroshot_rubric_radar.pdf



## Tables 9-12. Fine-tuned vs zero-shot Gemma 2 9B (test set, N = 510)

Grading metrics, off-topic detection under three conditions, confusion matrix and per-criterion results.

In [4]:
run('tables09_12_finetuning.py')

single-step test N 510 human flag 90
ZS multi-step: pm1=72.35 (369) macro=71.55 QWK=0.7610 r=0.7687 MAE=0.8745 bias=-0.2098
   Clarity      QWK=0.496 MAE=0.520 exact=53.5 bias=-0.324
   Terminology  QWK=0.663 MAE=0.376 exact=63.5 bias=-0.133
   Coverage     QWK=0.694 MAE=0.375 exact=64.9 bias=-0.178
   Accuracy     QWK=0.705 MAE=0.824 exact=44.1 bias=+0.216
Fine-tuned: pm1=78.43 (400) macro=78.95 QWK=0.8207 r=0.8220 MAE=0.7127 bias=+0.0853
   Clarity      QWK=0.702 MAE=0.312 exact=69.2 bias=+0.041
   Terminology  QWK=0.761 MAE=0.271 exact=73.1 bias=+0.059
   Coverage     QWK=0.734 MAE=0.310 exact=69.4 bias=+0.094
   Accuracy     QWK=0.790 MAE=0.576 exact=52.4 bias=-0.024
ZS multi-step: TP=89 FP=112 FN=1 TN=308 P=44.3 R=98.9 F1=61.2 incons=63/201=31.3
   LLM categories: {'On-Topic': 309, 'Off-Topic': 163, 'No Answer': 35, 'Insufficient answer - may have keywords but without any meaning to the required answer': 3}
Fine-tuned: TP=67 FP=12 FN=23 TN=408 P=84.8 R=74.4 F1=79.3 incons=0/79=0.0

## Table 13. Mean difference from the examiner grades and equivalence

Question-cluster 90% intervals (TOST margins ±0.25 and ±0.50) and 95% intervals for the fine-tuned model's metrics. The t-test and Wilcoxon lines are not reported in the paper.

In [5]:
run('table13_equivalence.py')

ZS multi-step: mean diff -0.2098 sd 1.2565 d=-0.167 t p=0.000182 W p=0.000898
   cluster 90%: mean -0.2098 [-0.3667, -0.0549] | 95%: mean -0.2098 [-0.4020, -0.0284]
Fine-tuned: mean diff +0.0853 sd 1.0372 d=0.082 t p=0.063862 W p=0.079098
   cluster 90%: mean +0.0853 [-0.0314, +0.1922] | 95%: mean +0.0853 [-0.0549, +0.2098]
FT 95% question-cluster CIs: ±1 [74.902 81.765] QWK [0.781 0.855] MAE [0.635 0.786]



## Power of the equivalence test

Simulated power of the question-cluster TOST for true differences of 0, 0.10, 0.15 and 0.20 points.

In [6]:
run('table13_power.py')

true bias 0.00: power ±0.25 = 95.6%, ±0.50 = 100.0%
true bias 0.10: power ±0.25 = 78.0%, ±0.50 = 100.0%
true bias 0.15: power ±0.25 = 47.2%, ±0.50 = 100.0%
true bias 0.20: power ±0.25 = 19.8%, ±0.50 = 100.0%



## Tables 14 and 16. Agreement among the four sets of human grades; multi-rater reliability

In [7]:
run('tables14_16_human_agreement.py')

Table 14: pairwise agreement among the four sets of human grades
  AE2 vs AE3         QWK 0.903  MAE 0.525  ±1 88.63%  bias +0.134
  Original vs AE2    QWK 0.852  MAE 0.660  ±1 80.59%  bias -0.052
  Original vs AE3    QWK 0.819  MAE 0.745  ±1 76.08%  bias +0.082
  Original vs AE1    QWK 0.788  MAE 0.834  ±1 75.88%  bias +0.234
  AE1 vs AE3         QWK 0.782  MAE 0.805  ±1 76.08%  bias -0.152
  AE1 vs AE2         QWK 0.780  MAE 0.802  ±1 75.10%  bias -0.286
  Model vs Original  QWK 0.821  MAE 0.713  ±1 78.43%

Table 16: multi-rater reliability (final grade 0-5)
  AE1-AE3 (primary)                ICC(2,1) 0.822 [0.758 0.871]  alpha interval 0.822 [0.757 0.871]  alpha ordinal 0.792
  Original + AE1-AE3 (secondary)   ICC(2,1) 0.821 [0.767 0.862]  alpha interval 0.821 [0.767 0.862]  alpha ordinal 0.786



## Table 15. Per-criterion QWK, human graders and the fine-tuned model

In [8]:
run('table15_percriterion_human.py')

Criterion   Clarity  Terminology  Coverage  Accuracy  Total
Model-Orig    0.702        0.761     0.734     0.790  0.821
Orig-AE1      0.586        0.609     0.682     0.708  0.788
Orig-AE2      0.684        0.719     0.771     0.816  0.852
Orig-AE3      0.662        0.698     0.730     0.781  0.819
AE1-AE2       0.555        0.625     0.692     0.729  0.780
AE1-AE3       0.639        0.596     0.671     0.732  0.782
AE2-AE3       0.731        0.816     0.834     0.879  0.903
Human min     0.555        0.596     0.671     0.708  0.780
Human max     0.731        0.816     0.834     0.879  0.903
Model-AE1     0.655        0.639     0.697     0.706  0.780
Model-AE2     0.733        0.727     0.807     0.822  0.847
Model-AE3     0.703        0.740     0.769     0.824  0.841



## Tables 17-18. All conditions and the nine paired comparisons

Fine-tuned model minus each condition; `SIG` = p < 0.05/9.

In [9]:
run('tables17_18_commercial.py')

=== Table metrics (±1 %, QWK 0-10, MAE 0-5, bias model-human 0-5) ===
Gemma FT         ±1  78.43  QWK 0.8207  MAE 0.7127  bias +0.0853  n=510
Gemma ZS         ±1  72.35  QWK 0.7610  MAE 0.8745  bias -0.2098  n=510
GPT-4o           ±1  71.57  QWK 0.7657  MAE 0.8451  bias -0.4039  n=510
GPT-5.2          ±1  68.04  QWK 0.7708  MAE 0.8971  bias +0.6716  n=510
Claude Sonnet 4  ±1  77.45  QWK 0.7867  MAE 0.8127  bias -0.2167  n=510
Claude Opus 4.6  ±1  77.06  QWK 0.8126  MAE 0.7490  bias -0.1490  n=510
BERT s42         ±1  73.14  QWK 0.7844  MAE 0.8333  bias +0.2098  n=510
BERT s43         ±1  76.27  QWK 0.7950  MAE 0.7804  bias +0.1647  n=510
BERT s44         ±1  75.10  QWK 0.7961  MAE 0.7931  bias +0.3637  n=510
FS GPT-4o        ±1  74.31  QWK 0.8069  MAE 0.7814  bias +0.2931  n=510
FS GPT-5.2       ±1  67.84  QWK 0.7613  MAE 0.9000  bias +0.7216  n=510
FS Opus 4.6      ±1  80.78  QWK 0.8351  MAE 0.7020  bias +0.1471  n=510
BERT mean [74.8366  0.7919  0.8023] min [73.1373  0.7844  0.7804] 

## Off-topic detection of the prompted conditions (descriptive)

In [10]:
run('offtopic_fewshot.py')

model          TP   FP   FN precision  recall
FT Gemma       67   12   23     0.848   0.744
ZS Gemma       89  112    1     0.443   0.989
GPT-4o ZS      80   49   10     0.620   0.889
GPT-4o FS      79   40   11     0.664   0.878
GPT-5.2 ZS     76   28   14     0.731   0.844
GPT-5.2 FS     72   23   18     0.758   0.800
Opus 4.6 ZS    71   23   19     0.755   0.789
Opus 4.6 FS    69   24   21     0.742   0.767



## Table 19. Disagreement patterns of the fine-tuned model

In [11]:
run('table19_error_analysis.py')

FP (on-topic flagged)          12  2.4%
FN (off-topic missed)          23  4.5%
Over (d >= +2)                 33  6.5%
Under (d <= -2)                24  4.7%
Partial (0.5 < |d| <= 1)       86  16.9%
AccConf (|gapAcc| >= 2)        45  8.8%
CovOver (gapCov >= 2)           2  0.4%

any disagreement |d|>0: 329 | exact 181 | |d|=0.5: 133 | |d|>1: 110
Over (d >= +2) both on-topic n=22 (all 33): {'Clarity': np.float64(1.0), 'Terminology': np.float64(0.86), 'Coverage': np.float64(1.09), 'Accuracy': np.float64(1.77)} | all cases: {'Clarity': np.float64(1.0), 'Terminology': np.float64(0.91), 'Coverage': np.float64(1.06), 'Accuracy': np.float64(1.67)}
Under (d <= -2) both on-topic n=22 (all 24): {'Clarity': np.float64(-0.91), 'Terminology': np.float64(-0.95), 'Coverage': np.float64(-0.86), 'Accuracy': np.float64(-1.86)} | all cases: {'Clarity': np.float64(-0.92), 'Terminology': np.float64(-0.96), 'Coverage': np.float64(-0.88), 'Accuracy': np.float64(-1.83)}

Partial-credit: |d| values {1.0: 86}

## Table D.22. Ablation runs (validation set) and per-epoch losses

In [12]:
run('tableD22_ablation.py')

r8              acc1 74.12 [68.0,80.0]  MAE 0.855 [0.710,1.007]  QWK 0.7627 [0.671,0.835]
LATEX $r = 8$ & 74.1 [68.0, 80.0] & 0.855 [0.710, 1.007] & 0.763 [0.671, 0.835]
r16             acc1 76.67 [71.2,81.8]  MAE 0.761 [0.641,0.886]  QWK 0.8122 [0.750,0.861]
LATEX $r = 16$ & 76.7 [71.2, 81.8] & 0.761 [0.641, 0.886] & 0.812 [0.750, 0.861]
r32             acc1 77.06 [71.2,82.5]  MAE 0.761 [0.642,0.884]  QWK 0.8043 [0.743,0.857]
LATEX $r = 32$ & 77.1 [71.2, 82.5] & 0.761 [0.642, 0.884] & 0.804 [0.743, 0.857]
r64             acc1 76.86 [71.2,82.0]  MAE 0.748 [0.636,0.862]  QWK 0.8078 [0.748,0.857]
LATEX $r = 64$ & 76.9 [71.2, 82.0] & 0.748 [0.636, 0.862] & 0.808 [0.748, 0.857]
r80             acc1 78.43 [73.7,82.9]  MAE 0.735 [0.628,0.841]  QWK 0.8141 [0.761,0.860]
LATEX $r = 80$\textsuperscript{*} & 78.4 [73.7, 82.9] & 0.735 [0.628, 0.841] & 0.814 [0.761, 0.860]
r96             acc1 76.47 [71.8,81.0]  MAE 0.770 [0.663,0.872]  QWK 0.8045 [0.754,0.850]
LATEX $r = 96$ & 76.5 [71.8, 81.0] & 

In [13]:
run('tableD22_losses.py')

        LR = 1$\times$10$^{-6}$ & 1.523 & 1.396 & 0.721 & 0.697 & 0.627 & 0.642 \\
        LR = 5$\times$10$^{-6}$\textsuperscript{*} & 0.469 & 0.477 & 0.327 & 0.472 & 0.270 & 0.489 \\
        LR = 1$\times$10$^{-5}$ & 0.360 & 0.467 & 0.142 & 0.658 & 0.126 & 0.670 \\
        LR = 2$\times$10$^{-5}$ & 0.166 & 0.619 & 0.081 & 0.638 & 0.068 & 0.659 \\



## Table E.23. Per-question QWK of the ten conditions

In [14]:
run('tableE23_per_question.py')

columns: ['Gemma FT', 'Gemma ZS', 'GPT-4o', 'GPT-5.2', 'Claude Sonnet 4', 'Claude Opus 4.6', 'FS GPT-4o', 'FS GPT-5.2', 'FS Opus 4.6', 'DeBERTa s42']
exact ties: []
        Q4 & \textbf{0.668} & 0.658 & 0.536 & 0.578 & 0.583 & 0.589 & 0.591 & 0.619 & 0.654 & 0.537 \\
        Q8 & 0.848 & 0.817 & 0.828 & 0.748 & 0.800 & 0.808 & 0.848 & 0.774 & 0.826 & \textbf{0.884} \\
        Q18 & 0.787 & 0.713 & 0.673 & 0.831 & \textbf{0.852} & 0.813 & 0.846 & 0.842 & 0.848 & 0.846 \\
        Q19 & 0.719 & 0.793 & 0.855 & 0.828 & \textbf{0.888} & 0.868 & 0.862 & 0.757 & 0.847 & 0.862 \\
        Q21 & 0.873 & 0.910 & 0.828 & 0.859 & \textbf{0.921} & 0.898 & 0.802 & 0.849 & 0.905 & 0.791 \\
        Q22 & \textbf{0.891} & 0.486 & 0.758 & 0.844 & 0.776 & 0.734 & 0.776 & 0.811 & 0.822 & 0.744 \\
        Q25 & 0.892 & 0.827 & 0.851 & 0.891 & 0.797 & 0.895 & \textbf{0.928} & 0.894 & 0.925 & 0.921 \\
        Q31 & 0.795 & 0.754 & 0.789 & 0.639 & 0.744 & \textbf{0.882} & 0.824 & 0.637 & 0.870 & 0.862 \\
     